# Bank Marketing Veri Tanıma

Bu not defteri hafta 2 için veri setinin ilk incelemesini yapar.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # sonraki hücreler 'reports/...' gibi proje köküne göreli yollar kullanır

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.data_loader import load_raw

df = load_raw()
print('Shape:', df.shape)
print(df.dtypes)
print(df.head())


Shape: (45211, 17)
age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object
   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1   

In [2]:
numeric_desc = df.describe(include=[np.number]).T
print(numeric_desc)


            count         mean          std     min    25%    50%     75%  \
age       45211.0    40.936210    10.618762    18.0   33.0   39.0    48.0   
balance   45211.0  1362.272058  3044.765829 -8019.0   72.0  448.0  1428.0   
day       45211.0    15.806419     8.322476     1.0    8.0   16.0    21.0   
duration  45211.0   258.163080   257.527812     0.0  103.0  180.0   319.0   
campaign  45211.0     2.763841     3.098021     1.0    1.0    2.0     3.0   
pdays     45211.0    40.197828   100.128746    -1.0   -1.0   -1.0    -1.0   
previous  45211.0     0.580323     2.303441     0.0    0.0    0.0     0.0   

               max  
age           95.0  
balance   102127.0  
day           31.0  
duration    4918.0  
campaign      63.0  
pdays        871.0  
previous     275.0  


In [3]:
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
for col in categorical_cols:
    print(f'\n{col}: {df[col].nunique()} unique values')
    print(df[col].value_counts().head(10).to_string())



job: 12 unique values
job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240

marital: 3 unique values
marital
married     27214
single      12790
divorced     5207

education: 4 unique values
education
secondary    23202
tertiary     13301
primary       6851
unknown       1857

default: 2 unique values
default
no     44396
yes      815

housing: 2 unique values
housing
yes    25130
no     20081

loan: 2 unique values
loan
no     37967
yes     7244

contact: 3 unique values
contact
cellular     29285
unknown      13020
telephone     2906

month: 12 unique values
month
may    13766
jul     6895
aug     6247
jun     5341
nov     3970
apr     2932
feb     2649
jan     1403
oct      738
sep      579

poutcome: 4 unique values
poutcome
unknown    36959
failure     4901
other       1840
success     1511

y: 2 unique values
y

In [4]:
target_counts = df['y'].value_counts()
target_pct = df['y'].value_counts(normalize=True) * 100
print('Target counts:', target_counts.to_dict())
print('Target percentages:', target_pct.to_dict())
fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind='bar', ax=ax, color=['#4C72B0', '#55A868'])
ax.set_title('Hedef Sınıf Dağılımı (y)', fontsize=12)
ax.set_xlabel('Sınıf')
ax.set_ylabel('Kayıt sayısı')
ax.set_xticklabels(['no', 'yes'], rotation=0)
fig.tight_layout()
fig.savefig('reports/figures/h2_hedef_dagilimi.png', dpi=150)
plt.close(fig)


Target counts: {'no': 39922, 'yes': 5289}
Target percentages: {'no': 88.30151954170445, 'yes': 11.698480458295547}


In [5]:
unknown_counts = {col: int((df[col] == 'unknown').sum()) for col in df.columns}
unknown_series = pd.Series(unknown_counts).sort_values(ascending=False, kind='stable')
print('Unknown counts:', unknown_series.to_dict())
fig, ax = plt.subplots(figsize=(10, 6))
unknown_series.plot(kind='barh', ax=ax, color='#E76F51')
ax.set_title('Her Sütundaki "unknown" Değerleri', fontsize=12)
ax.set_xlabel('Sayı')
ax.set_ylabel('Değişken')
fig.tight_layout()
fig.savefig('reports/figures/h2_unknown_dagilimi.png', dpi=150)
plt.close(fig)


Unknown counts: {'poutcome': 36959, 'contact': 13020, 'education': 1857, 'job': 288, 'age': 0, 'marital': 0, 'default': 0, 'balance': 0, 'housing': 0, 'loan': 0, 'day': 0, 'month': 0, 'duration': 0, 'campaign': 0, 'pdays': 0, 'previous': 0, 'y': 0}


In [6]:
pdays_minus_one = int((df['pdays'] == -1).sum())
duplicate_rows = int(df.duplicated().sum())
duration_by_target = df.groupby('y')['duration'].mean()
from src.eda import CROSSTAB_TOTAL_LABEL, PDAYS_CONTACTED_ROW, PDAYS_NOT_CONTACTED_ROW, pdays_poutcome_crosstab
pdays_poutcome = pdays_poutcome_crosstab(df)
print('pdays_poutcome:\n', pdays_poutcome)
print('pdays == -1:', pdays_minus_one)
print('duplicate rows:', duplicate_rows)
print('duration_by_target:', duration_by_target.to_dict())


pdays_poutcome:
 poutcome               failure  other  success  unknown    All
pdays                                                         
pdays_equal_minus_one        0      0        0    36954  36954
pdays_not_minus_one       4901   1840     1511        5   8257
All                       4901   1840     1511    36959  45211
pdays == -1: 36954
duplicate rows: 0
duration_by_target: {'no': 221.18280647262162, 'yes': 537.2945736434109}


In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from src.config import DROP_COLS, RANDOM_STATE, TARGET, TEST_SIZE
X = df.drop(columns=DROP_COLS + [TARGET])
y = df[TARGET].map({'yes': 1, 'no': 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
baseline_model = DummyClassifier(strategy='most_frequent')
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)
baseline_proba = baseline_model.predict_proba(X_test)[:, 1]
metrics = {
    'accuracy': accuracy_score(y_test, baseline_pred),
    'precision': precision_score(y_test, baseline_pred, zero_division=0),
    'recall': recall_score(y_test, baseline_pred, zero_division=0),
    'f1': f1_score(y_test, baseline_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, baseline_proba),
    'pr_auc': average_precision_score(y_test, baseline_proba),
}
print(metrics)


{'accuracy': 0.8830034280659074, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'roc_auc': 0.5, 'pr_auc': 0.11699657193409267}


In [8]:
summary = {
    'dataset_shape': {'rows': int(df.shape[0]), 'cols': int(df.shape[1])},
    'target_counts': {str(k): int(v) for k, v in target_counts.items()},
    'target_pct': {str(k): round(float(v), 4) for k, v in target_pct.items()},
    'unknown_counts': {str(k): int(v) for k, v in unknown_series.items()},
    'pdays_minus_one_count': pdays_minus_one,
    'pdays_minus_one_pct': round((pdays_minus_one / len(df)) * 100, 4),
    'pdays_poutcome_crosstab': {
        'pdays_equal_minus_one': {str(k): int(v) for k, v in pdays_poutcome.loc[PDAYS_NOT_CONTACTED_ROW].to_dict().items()},
        'pdays_not_minus_one': {str(k): int(v) for k, v in pdays_poutcome.loc[PDAYS_CONTACTED_ROW].to_dict().items()},
        'all': {str(k): int(v) for k, v in pdays_poutcome.loc[CROSSTAB_TOTAL_LABEL].to_dict().items()},
    },
    'pdays_minus_one_poutcome_unknown_count': int(((df['pdays'] == -1) & (df['poutcome'] == 'unknown')).sum()),
    'pdays_not_minus_one_poutcome_unknown_count': int(((df['pdays'] != -1) & (df['poutcome'] == 'unknown')).sum()),
    'previous_zero_equals_pdays_minus_one': bool((df['previous'] == 0).equals(df['pdays'] == -1)),
    'baseline_model_metrics': {k: round(float(v), 4) for k, v in metrics.items()},
    'duplicate_rows': duplicate_rows,
    'duration_mean_by_target': {str(k): round(float(v), 2) for k, v in duration_by_target.items()},
}
json_path = Path('reports/haftalik/hafta2_ozet.json')
json_path.parent.mkdir(parents=True, exist_ok=True)
json_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json_path)


reports/haftalik/hafta2_ozet.json


In [9]:
glossary_lines = ['# Veri Sözlüğü', '', '| Değişken | Tür | Açıklama | Olası değerler veya aralık | "unknown" içeriyor mu? | Modelde kullanılacak mı? |', '| --- | --- | --- | --- | --- | --- |']
for col in df.columns:
    is_numeric = pd.api.types.is_numeric_dtype(df[col])
    if is_numeric:
        dtype_label = 'sayısal'
        if col == 'duration':
            description = 'Müşterinin arama sırasında konuşma süresi. Hedefe çok yakın olduğundan modelde kullanılmaz.'
            values = f'{int(df[col].min())} - {int(df[col].max())}'
            model_use = 'hayır (veri sızıntısı riski)'
        else:
            description = f'{col} alanı için gözlenen değer aralığıdır.'
            values = f'{int(df[col].min())} - {int(df[col].max())}'
            model_use = 'evet'
    else:
        dtype_label = 'kategorik'
        unique_values = sorted(df[col].dropna().astype(str).unique().tolist())
        description = 'Kategorik özellik; müşteri profilini tanımlar.'
        values = ', '.join(unique_values[:10]) + (' ...' if len(unique_values) > 10 else '')
        model_use = 'evet'
    has_unknown = bool((df[col].astype(str) == 'unknown').sum() > 0)
    glossary_lines.append(f'| {col} | {dtype_label} | {description} | {values} | {str(has_unknown).lower()} | {model_use} |')
Path('docs/veri_sozlugu.md').write_text('\n'.join(glossary_lines) + '\n', encoding='utf-8')
print('docs/veri_sozlugu.md created')


docs/veri_sozlugu.md created
